# 🌸 Task 1: Iris Flower Classification
**Dataset:** IRIS.csv from amankharwal/Website-data

Classify Iris species (setosa, versicolor, virginica) using sepal/petal measurements.

In [ ]:
# Install if needed
# !pip install scikit-learn pandas matplotlib seaborn

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')
print("Libraries imported!")

## 2. Load the Real Dataset

In [ ]:
# Load directly from GitHub (the official dataset link)
url = "https://raw.githubusercontent.com/amankharwal/Website-data/master/IRIS.csv"
df = pd.read_csv(url)

# OR load local file:
# df = pd.read_csv("iris.csv")

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("Species distribution:")
print(df['species'].value_counts())
print("\nMissing values:", df.isnull().sum().sum())
print("\nBasic stats:")
df.describe()

## 3. Exploratory Data Analysis

In [ ]:
species_list = df['species'].unique()
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for i, ax in enumerate(axes.flat):
    for j, sp in enumerate(species_list):
        ax.hist(df[df['species']==sp][features[i]], alpha=0.7, label=sp, color=colors[j], bins=15)
    ax.set_xlabel(features[i]); ax.set_ylabel('Count')
    ax.set_title(f'Distribution: {features[i]}'); ax.legend(fontsize=8)
plt.suptitle('Feature Distributions by Species', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('iris_distributions.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
sns.pairplot(df, hue='species', palette=dict(zip(species_list, colors)), height=2.3)
plt.suptitle('Iris Pairplot', y=1.02, fontsize=13, fontweight='bold')
plt.savefig('iris_pairplot.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[features].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('iris_heatmap.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. Data Preprocessing

In [ ]:
X = df[features].values
le = LabelEncoder()
y = le.fit_transform(df['species'])

print("Classes:", le.classes_)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 5. Train & Compare Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=200, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':                 SVC(kernel='rbf', C=1.0, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    acc    = accuracy_score(y_test, y_pred)
    cv     = cross_val_score(model, X_train_s, y_train, cv=5)
    results[name] = {'Accuracy': acc, 'CV Mean': cv.mean(), 'CV Std': cv.std()}
    print(f"{name:<25} Test Acc: {acc:.4f}  CV: {cv.mean():.4f} ± {cv.std():.4f}")

results_df = pd.DataFrame(results).T
print("\nResults Summary:")
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(results_df))
bars = ax.bar(x, results_df['Accuracy'], color=['#FF6B6B','#4ECDC4','#45B7D1','#FFA07A'], width=0.5)
ax.set_xticks(x); ax.set_xticklabels(results_df.index, rotation=10)
ax.set_ylim(0.9, 1.02); ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy Comparison', fontsize=13, fontweight='bold')
for bar, v in zip(bars, results_df['Accuracy']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout(); plt.savefig('iris_model_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. Best Model — Detailed Report

In [ ]:
best_name  = results_df['Accuracy'].idxmax()
best_model = models[best_name]
y_pred     = best_model.predict(X_test_s)

print(f"Best Model : {best_name}")
print(f"Accuracy   : {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('iris_confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. Feature Importance (Random Forest)

In [ ]:
rf  = models['Random Forest']
imp = pd.Series(rf.feature_importances_, index=features).sort_values()
plt.figure(figsize=(8, 4))
imp.plot(kind='barh', color='#45B7D1')
plt.xlabel('Importance'); plt.title('Feature Importances — Random Forest', fontweight='bold')
plt.tight_layout(); plt.savefig('iris_feature_importance.png', dpi=150, bbox_inches='tight'); plt.show()

## 8. Predict on New Sample

In [ ]:
sample = np.array([[5.1, 3.5, 1.4, 0.2]])
pred   = best_model.predict(scaler.transform(sample))
print(f"Input     : {sample[0]}")
print(f"Predicted : {le.inverse_transform(pred)[0].upper()}")